# Wire Formats and Version Conversion

The rest of iso8583sim represents a message as a string: an ASCII MTI, a hex bitmap, and binary fields as hex. Real hosts exchange **bytes**, often with a binary bitmap, packed BCD numbers or EBCDIC text.

This notebook shows how to encode and decode those layouts, then how to convert messages between the 1987, 1993 and 2003 versions.

In [1]:
import sys

sys.path.insert(0, "..")  # For development - remove in production

In [2]:
import copy

from iso8583sim.core.builder import ISO8583Builder
from iso8583sim.core.parser import ISO8583Parser
from iso8583sim.core.samples import sample_message
from iso8583sim.wire import Encoding, WireFormat, decode_message, encode_message

message = sample_message()  # a VISA authorization request
text = ISO8583Builder().build(copy.deepcopy(message))
print(text)

01007024058000C08000164111111111111111000000000000001000123456261205100100TERM0001MERCHANT123456 840


## The same message in each preset

In [3]:
for preset in ["ascii-hex", "ascii-binary", "bcd", "ebcdic"]:
    data = encode_message(copy.deepcopy(message), WireFormat.preset(preset))
    print(f"{preset:13} {len(data):3} bytes  {data[:24].hex(' ').upper()} ...")

ascii-hex     100 bytes  30 31 30 30 37 30 32 34 30 35 38 30 30 30 43 30 38 30 30 30 31 36 34 31 ...
ascii-binary   92 bytes  30 31 30 30 70 24 05 80 00 C0 80 00 31 36 34 31 31 31 31 31 31 31 31 31 ...
bcd            63 bytes  01 00 70 24 05 80 00 C0 80 00 16 41 11 11 11 11 11 11 11 00 00 00 00 00 ...
ebcdic         92 bytes  F0 F1 F0 F0 70 24 05 80 00 C0 80 00 F1 F6 F4 F1 F1 F1 F1 F1 F1 F1 F1 F1 ...


`ascii-hex` is byte for byte what `build()` returns, so existing string messages convert losslessly:

In [4]:
encode_message(copy.deepcopy(message), WireFormat.ascii_hex()) == text.encode("ascii")

True

## Reading a BCD message

In BCD, the MTI `0100` is two bytes (`01 00`), the bitmap is 8 raw bytes, and each pair of digits shares a byte. The PAN is prefixed by a one-byte length (`16`):

In [5]:
data = ISO8583Builder().build_bytes(copy.deepcopy(message), WireFormat.bcd())
print("MTI    ", data[:2].hex(" ").upper())
print("bitmap ", data[2:10].hex(" ").upper())
print("PAN    ", data[10:19].hex(" ").upper())

decoded = ISO8583Parser().parse_bytes(data, WireFormat.bcd())
print("\nDecoded:", decoded.mti, decoded.network, decoded.fields[2], decoded.fields[4])

MTI     01 00
bitmap  70 24 05 80 00 C0 80 00
PAN     16 41 11 11 11 11 11 11 11

Decoded: 0100 CardNetwork.VISA 4111111111111111 000000001000


Decoded messages have exactly the field values the string parser gives, so validation, explanation and conversion all work on them.

In [6]:
decoded.fields == ISO8583Parser().parse(text).fields

True

## A custom layout

Each part can be chosen independently, for example BCD numbers with 2-byte binary length prefixes and a trailing `F` for odd-length fields:

In [7]:
custom = WireFormat(
    mti=Encoding.BCD,
    length_prefix=Encoding.BINARY,
    numeric=Encoding.BCD,
    bcd_odd_padding="right",
)
amex = sample_message(pan="378282246310005")  # 15-digit PAN, so it needs a filler nibble
data = encode_message(copy.deepcopy(amex), custom)
print(data.hex(" ").upper())
print(decode_message(data, custom).fields[2])

01 00 70 24 05 80 00 C0 80 00 0F 37 82 82 24 63 10 00 5F 00 00 00 00 00 00 00 10 00 12 34 56 26 12 00 51 00 01 00 54 45 52 4D 30 30 30 31 4D 45 52 43 48 41 4E 54 31 32 33 34 35 36 20 08 40
378282246310005


## Converting between versions

`convert_message` rewrites the MTI version digit, moves original data elements between field 90 (1987) and field 56 (2003), and reports anything it can't carry over instead of truncating it.

In [8]:
from iso8583sim.core.convert import convert_message
from iso8583sim.core.types import ISO8583Message, ISO8583Version

reversal = ISO8583Message(
    mti="0400",
    fields={
        2: "4111111111111111",
        3: "000000",
        4: "000000001000",
        11: "123456",
        52: "2A3D408A1977DDE9",  # an 8-byte PIN block
        57: "ABC",
        90: "010012345612251030000000001234500000000000",
    },
)
result = convert_message(reversal, ISO8583Version.V2003)

print("MTI:", reversal.mti, "->", result.message.mti)
print("Field 56:", result.message.fields[56])
print("\nNotes:")
for note in result.notes:
    print(" -", note)
print("\nDropped:")
for field, reason in result.dropped.items():
    print(f" - field {field}: {reason}")

MTI: 0400 -> 2400
Field 56: 010012345612251030000000001234500000000000

Notes:
 - Moved original data elements from field 90 to field 56.
 - Field 57 is 'Authorization Life Cycle Code (2003)' in 2003 (was 'Reserved National'). Check the value still means the same thing.

Dropped:
 - field 52: PIN data must be re-encrypted for the target version's PIN block size


Next: [11_network_simulation.ipynb](11_network_simulation.ipynb) sends these bytes over TCP.